# 01: Setup, Authentication & AST Corpus Exploration
### Mechanistic Interpretability of Native MLP Monosemanticity in `google/gemma-4-12B-it`
**Gemma 4 Paper Track**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baddest-cmd/gemma-clean-code/blob/main/notebooks/01_setup_and_corpus.ipynb)

This notebook establishes the foundational experimental pipeline:
1. **Environment Initialization & Dependencies**: Automatic detection of Google Colab vs. local workspace.
2. **Hugging Face Authentication**: Secure token resolution via Colab Secrets (`HF_TOKEN`) or local `.env`.
3. **Corpus Ingestion & Analysis**: Inspection of the contrastive AST dataset across 5 syntactic categories (`try_except`, `with_statement`, `async_await`, `class_def`, `nested_loop`).

## 1. Environment & Repository Setup

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("⚡ Running in Google Colab environment.")
    if not os.path.exists("gemma-clean-code"):
        !git clone https://github.com/baddest-cmd/gemma-clean-code.git
        %cd gemma-clean-code
    else:
        %cd gemma-clean-code
        !git pull origin main
    !pip install -q -r requirements.txt
else:
    print(f"💻 Running in local development workspace: {os.getcwd()}")

# Ensure project root is in sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print(f"Project root added to sys.path: {project_root}")

## 2. Secure Hugging Face Authentication

We use `scripts/auth_utils.py` to authenticate. It automatically queries:
- Google Colab Secrets (`userdata.get('HF_TOKEN')`)
- Local environment / `.env` (`python-dotenv`)

In [ ]:
from scripts.auth_utils import setup_hf_auth

# Authenticate with Hugging Face Hub (gated access for Gemma models)
try:
    auth_token = setup_hf_auth()
    print(f"✅ Successfully authenticated with Hugging Face Hub! (Token prefix: {auth_token[:7]}...)")
except Exception as e:
    print(f"❌ Authentication failed: {e}")

## 3. Load & Ingest Synthetic AST Corpus

In [ ]:
import json
from pathlib import Path
import pandas as pd

corpus_path = Path(project_root) / "corpus" / "gemma_ast_corpus.jsonl"
assert corpus_path.exists(), f"Corpus file not found at: {corpus_path}"

records = []
with open(corpus_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

df = pd.DataFrame(records)
print(f"✅ Successfully loaded {len(df)} contrastive snippet pairs from {corpus_path.name}")
df[["pair_id", "category", "target_construct", "diff_summary", "target_char_len", "control_char_len"]].head(10)

## 4. Corpus Distribution & Category Breakdown

In [ ]:
summary = df.groupby("category").agg(
    pair_count=("pair_id", "count"),
    avg_target_len=("target_char_len", "mean"),
    avg_control_len=("control_char_len", "mean"),
    all_valid=("target_valid", "all"),
).reset_index()

print("Corpus Category Summary:")
display(summary)

## 5. Contrastive Pair Inspection Across Categories

Below we display representative contrastive pairs demonstrating the structural variation between Target constructs and Near-Miss controls.

In [ ]:
categories = df["category"].unique()

for cat in categories:
    sample = df[df["category"] == cat].iloc[0]
    print("=" * 75)
    print(f"Category: {cat.upper()} | Pair ID: {sample['pair_id']} | Construct: {sample['target_construct']}")
    print(f"Description: {sample['diff_summary']}")
    print("-" * 75)
    print("🟢 [TARGET CONSTRUCT CODE]:")
    print(sample["target_code"].strip())
    print("\n🔴 [NEAR-MISS CONTROL CODE]:")
    print(sample["control_code"].strip())
    print(f"\nTarget AST Nodes: {sample['target_ast_nodes']}")
    print(f"Control AST Nodes: {sample['control_ast_nodes']}\n")

## 6. Next Steps: Activation Caching in `google/gemma-4-12B-it`

In **Notebook 02 (`02_activation_caching.ipynb`)**, we will:
1. Load `google/gemma-4-12B-it` using `TransformerLens` (or `HookedTransformer`).
2. Hook intermediate MLP activations (`blocks.{layer}.mlp.hook_post`) across target and control sequences.
3. Compute differential activation heatmaps $\Delta A = A_{\text{target}} - A_{\text{control}}$ to isolate monosemantic AST neuron candidates.